# event_log 변환 SQL 생성

기존에 생성한 `*_logs.sql` 파일들(`first_save_history` 형식)을 읽어,
`event_log` 테이블에 맞는 INSERT문으로 변환합니다.

모든 이벤트 로그(login, logout 포함)는 `first_save_history` 테이블 하나에 저장되며,
각 row의 `json_log.event_name` 값으로 어떤 이벤트인지 구분합니다.

## event_log 테이블 컬럼
```
id, user_id, event_name, event_timestamp, product_id, product_name, product_category,
approved_amount, action_type, coupon_code, discount_amount, expiry_date,
search_keyword, page_name, dwell_time, review_rating, earned_points, earn_reason,
login_id, created_at, ad_id, client_uuid
```

## 매핑 규칙 (event_name별)
| event_name | 사용 컬럼 |
|---|---|
| login / logout | user_id, login_id, client_uuid, event_timestamp |
| page_view | + page_name, dwell_time |
| search_button_click | + search_keyword |
| product_detail_view | + product_id/name/category, dwell_time |
| purchase_button_click | + product_id/name/category, approved_amount |
| cart_click / wishlist_click | + product_id/name/category, action_type |
| ad_click / ad_exposure | + product_id/name/category, ad_id |
| coupon_received / coupon_used | + coupon_code, discount_amount, expiry_date(coupon_used는 NULL) |

- `id`는 SERIAL이므로 INSERT문에 포함하지 않음
- `review_rating`, `earned_points`, `earn_reason`은 사용하지 않는 컬럼 → 항상 NULL
- `created_at`은 원본 SQL의 history_timestamp 값 사용
- `login_id`는 json의 `user_login_id` 값 사용

## 사용 방법
1. 변환하고자 하는 `*_logs.sql` 파일들을 이 노트북과 같은 디렉토리에 둔다
2. `SOURCE_SQL_FILES` 리스트에 파일명을 추가한다
3. 노트북을 실행하면 `event_log_insert.sql`이 생성된다

In [1]:
import re
import json
import os

In [2]:
# ───────────────────────────────────────────
# 설정값: 변환할 원본 SQL 파일 목록
# (없는 파일은 자동으로 건너뜀)
# ───────────────────────────────────────────
SOURCE_SQL_FILES = [
    'login_logout_logs.sql',
    'page_view_logs.sql',
    'search_button_click_logs.sql',
    'product_detail_view_logs.sql',
    'purchase_button_click_logs.sql',
    'cart_click_logs.sql',
    'wishlist_click_logs.sql',
    'ad_click_logs.sql',
    'ad_exposure_logs.sql',
    'coupon_received_logs.sql',
    'coupon_used_logs.sql',
]

OUTPUT_SQL_FILE = 'event_log_insert.sql'

In [3]:
# event_log 테이블 컬럼 순서 (id 제외)
EVENT_LOG_COLUMNS = [
    'user_id', 'event_name', 'event_timestamp', 'product_id', 'product_name',
    'product_category', 'approved_amount', 'action_type', 'coupon_code',
    'discount_amount', 'expiry_date', 'search_keyword', 'page_name',
    'dwell_time', 'review_rating', 'earned_points', 'earn_reason',
    'login_id', 'created_at', 'ad_id', 'client_uuid'
]

In [4]:
def extract_rows_from_sql(sql_text):
    """
    INSERT INTO ... (history_timestamp, json_log) VALUES
      ('2026-06-01 00:00:01.000000', '{"event_name": ...}'),
      ...
    형식의 SQL 문자열에서 (history_timestamp, json_log) 튜플 리스트를 추출한다.
    json_log 내부의 '' (escaped single quote)는 '로 복원한다.
    """
    # 각 value 튜플: ('...', '...') 형태를 매칭
    # json_log 안에 ''(escaped ')가 있을 수 있으므로 non-greedy + 마지막 '를 기준으로 분리
    pattern = re.compile(
        r"\(\s*'([^']*(?:''[^']*)*)'\s*,\s*'((?:[^']|'')*)'\s*\)",
        re.DOTALL
    )

    rows = []
    for m in pattern.finditer(sql_text):
        history_ts_raw = m.group(1)
        json_log_raw   = m.group(2)

        # SQL escape 복원 ('' -> ')
        history_ts = history_ts_raw.replace("''", "'")
        json_log   = json_log_raw.replace("''", "'")

        rows.append((history_ts, json_log))

    return rows

In [5]:
def sql_literal(value):
    """파이썬 값을 SQL 리터럴 문자열로 변환 (None -> NULL, 문자열은 quote+escape)"""
    if value is None:
        return 'NULL'
    if isinstance(value, bool):
        return 'TRUE' if value else 'FALSE'
    if isinstance(value, (int, float)):
        return str(value)
    # 문자열: ' -> '' 이스케이프
    escaped = str(value).replace("'", "''")
    return f"'{escaped}'"

In [6]:
def map_to_event_log(history_ts, json_log):
    """
    history_timestamp + json_log(dict)를 event_log 테이블 컬럼 dict로 매핑한다.
    """
    data = json.loads(json_log)

    event_name = data.get('event_name')

    row = {col: None for col in EVENT_LOG_COLUMNS}

    # ── 공통 ──
    row['user_id']         = data.get('user_id')
    row['event_name']      = event_name
    row['event_timestamp'] = data.get('event_timestamp')
    row['login_id']        = data.get('user_login_id')
    row['created_at']      = history_ts
    row['client_uuid']     = data.get('client_uuid')

    # ── 이벤트별 매핑 ──
    if event_name in ('login', 'logout'):
        pass  # 공통 필드만 사용

    elif event_name == 'page_view':
        row['page_name']  = data.get('pageName')
        row['dwell_time'] = data.get('dwellTime')

    elif event_name == 'search_button_click':
        row['search_keyword'] = data.get('searchKeyword')

    elif event_name == 'product_detail_view':
        row['product_id']       = data.get('productId')
        row['product_name']     = data.get('productName')
        row['product_category'] = data.get('productCategory')
        row['dwell_time']       = data.get('dwellTime')

    elif event_name == 'purchase_button_click':
        row['product_id']       = data.get('productId')
        row['product_name']     = data.get('productName')
        row['product_category'] = data.get('productCategory')
        row['approved_amount']  = data.get('approvedAmount')

    elif event_name in ('cart_click', 'wishlist_click'):
        row['product_id']       = data.get('productId')
        row['product_name']     = data.get('productName')
        row['product_category'] = data.get('productCategory')
        row['action_type']      = data.get('actionType')

    elif event_name in ('ad_click', 'ad_exposure'):
        row['product_id']       = data.get('productId')
        row['product_name']     = data.get('productName')
        row['product_category'] = data.get('productCategory')
        row['ad_id']            = data.get('adId')

    elif event_name in ('coupon_received', 'coupon_used'):
        row['coupon_code']     = data.get('couponCode')
        row['discount_amount'] = data.get('discountAmount')
        row['expiry_date']     = data.get('expiryDate')  # coupon_used에는 없으므로 None

    else:
        # 알 수 없는 이벤트는 공통 필드만 채워서 그대로 둠
        pass

    return row

In [7]:
all_event_rows = []

for filename in SOURCE_SQL_FILES:
    if not os.path.exists(filename):
        print(f'⚠️  {filename} 없음 → 건너뜀')
        continue

    with open(filename, 'r', encoding='utf-8') as f:
        sql_text = f.read()

    raw_rows = extract_rows_from_sql(sql_text)

    converted = [map_to_event_log(history_ts, json_log) for history_ts, json_log in raw_rows]
    all_event_rows.extend(converted)

    print(f'✅ {filename} → {len(raw_rows)}건 변환')

print(f'\n총 {len(all_event_rows)}건 변환 완료')

✅ login_logout_logs.sql → 500건 변환
✅ page_view_logs.sql → 500건 변환
✅ search_button_click_logs.sql → 500건 변환
✅ product_detail_view_logs.sql → 500건 변환
✅ purchase_button_click_logs.sql → 300건 변환
✅ cart_click_logs.sql → 500건 변환
✅ wishlist_click_logs.sql → 500건 변환
✅ ad_click_logs.sql → 500건 변환
✅ ad_exposure_logs.sql → 500건 변환
✅ coupon_received_logs.sql → 500건 변환
✅ coupon_used_logs.sql → 150건 변환

총 4950건 변환 완료


In [8]:
# event_log INSERT SQL 생성
columns_str = ', '.join(EVENT_LOG_COLUMNS)

lines  = [f'INSERT INTO event_log ({columns_str}) VALUES']
values = []

for row in all_event_rows:
    literals = [sql_literal(row[col]) for col in EVENT_LOG_COLUMNS]
    values.append('  (' + ', '.join(literals) + ')')

lines.append(',\n'.join(values) + ';')
event_log_sql = '\n'.join(lines)

with open(OUTPUT_SQL_FILE, 'w', encoding='utf-8') as f:
    f.write(event_log_sql)

print(f'✅ {len(all_event_rows)}건 event_log INSERT SQL 생성 완료 → {OUTPUT_SQL_FILE}')

✅ 4950건 event_log INSERT SQL 생성 완료 → event_log_insert.sql


In [9]:
# ── 미리보기 ──
print('=== EVENT_LOG INSERT SQL (앞 1000자) ===')
print(event_log_sql[:1000])

=== EVENT_LOG INSERT SQL (앞 1000자) ===
INSERT INTO event_log (user_id, event_name, event_timestamp, product_id, product_name, product_category, approved_amount, action_type, coupon_code, discount_amount, expiry_date, search_keyword, page_name, dwell_time, review_rating, earned_points, earn_reason, login_id, created_at, ad_id, client_uuid) VALUES
  (44, 'login', '2025-07-29T22:13:34.000+09:00', NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 'user0044', '2025-07-29 22:13:35.000000', NULL, 'a6a86964-a0fc-4a50-b73c-3f41a2fa190c'),
  (44, 'logout', '2025-07-29T23:25:59.000+09:00', NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 'user0044', '2025-07-29 23:26:00.000000', NULL, 'a6a86964-a0fc-4a50-b73c-3f41a2fa190c'),
  (30, 'login', '2026-05-15T16:56:46.000+09:00', NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 'user0030', '2026-05-15 16:56:47.000000', NULL, '9e632748-5789-4139-8a78-8